In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
import os
from PIL import Image

# ====================== THAY ĐƯỜNG DẪN ======================
TEST_DIR = "C:/DUT/Ki 2/Edge AI/dataset/test"   # ←←← THAY
MODEL_PATH = "model_quant_int8.tflite"  # upload .tflite từ Notebook 1

# Load TFLite interpreter
interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Input phải là int8
input_scale, input_zero_point = input_details[0]["quantization"]

test_images = sorted([f for f in os.listdir(TEST_DIR) if f.endswith(('.jpg','.png'))])

predictions = []
for img_name in test_images:
    # Đọc và preprocess đúng chuẩn 32x32 RGB
    img = Image.open(os.path.join(TEST_DIR, img_name)).resize((32, 32))
    img_array = np.array(img, dtype=np.float32) / 255.0
    img_array = np.expand_dims(img_array, axis=0)          # (1,32,32,3)
    
    # Quantize sang int8
    img_int8 = (img_array / input_scale + input_zero_point).astype(np.int8)
    
    interpreter.set_tensor(input_details[0]['index'], img_int8)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])
    
    pred_class = np.argmax(output[0])
    predictions.append(pred_class)

# ====================== TẠO SUBMISSION ======================
ids = [f.split('.')[0] for f in test_images]   # 00000, 00001...
submission = pd.DataFrame({'Id': ids, 'Label': predictions})
submission.to_csv("submission.csv", index=False)

print("✅ File submission.csv đã tạo!")
submission.head()

✅ File submission.csv đã tạo!


,Id,Label
0,00000,4
1,00007,2
2,00012,2
3,00014,6
4,00022,1
